# OpenArm 違和感検知: Day 1 確認(Kaggle GPU)

PLAN.md 第 6 節の 4・5 を確認する。

- 4: Kaggle 環境(ヘッドレス描画、JAX/MJX の GPU 動作、mp4 書き出し)
- 5: MJX の並列速度から PPO の学習時間を見積もる

**設定**: Accelerator = GPU T4(x1 で可)、Internet = On。上から順に実行する。
結果は最後のセルで `/kaggle/working/day1_results.json` に保存される。

In [ ]:
!nvidia-smi

In [ ]:
%%capture pip_log
!pip install -q "jax[cuda12]==0.10.2" mujoco==3.17.0 mujoco-mjx==3.17.0 warp-lang==1.18.0 \
    brax==0.14.2 playground==0.2.0 flax==0.12.8 optax==0.2.8 mediapy==1.2.7 "imageio[ffmpeg]"

In [ ]:
print(pip_log.stdout[-3000:], pip_log.stderr[-3000:])

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"
os.chdir("/kaggle/working")
!rm -rf openarm_mujoco && git clone -q https://github.com/enactic/openarm_mujoco.git && \
    git -C openarm_mujoco checkout -q a3c8a57ec15d385e3c1dbd3fd090ee486b7f258c && git -C openarm_mujoco log -1 --format=%H
!mkdir -p src/openarm_anomaly outputs

In [ ]:
%%writefile src/openarm_anomaly/__init__.py

In [ ]:
%%writefile src/openarm_anomaly/scene.py
"""運搬タスク用のシーンを OpenArm v1 単腕モデルから組み立てる。

MJX 向けに、衝突用の高頂点メッシュ(指は 3 万頂点超)を削除し、
ハンドと指先パッドを箱プリミティブで置き換える。腕リンクは衝突を持たない。
"""
from pathlib import Path

import mujoco

ROOT = Path(__file__).resolve().parents[2]
OPENARM_XML = ROOT / "openarm_mujoco" / "v1" / "openarm.xml"

# 腕の土台を載せる台座の高さ。手首の可動域(joint6 ±45°)が狭く、
# 机と同じ高さに置くと上からの把持で届く範囲がほぼ無くなるため。
PEDESTAL_H = 0.15
# ステーション(机上面での物体中心の xy)。上からの把持で到達できる範囲から選んだ。
STATION_A = (0.30, -0.20)
STATION_B = (0.30, 0.20)

# 指パッド: 指ボディ座標系(z が指の伸びる向き、y が閉じる向き)。
# メッシュの内側面は y=+0.0058(右指)、先端は z=0.0805。
PAD_HALF = (0.012, 0.004, 0.015)
PAD_POS_Z = 0.065
PAD_INNER_Y = 0.0058


def build_spec(box_half=0.03, box_mass=0.5, friction=1.0, timestep=0.002):
    spec = mujoco.MjSpec.from_file(str(OPENARM_XML))
    spec.option.timestep = timestep
    spec.option.cone = mujoco.mjtCone.mjCONE_PYRAMIDAL

    for g in list(spec.geoms):
        if g.name.endswith("_collision"):
            spec.delete(g)
    for mesh in list(spec.meshes):
        if mesh.name.endswith("_collision"):
            spec.delete(mesh)

    # 実機のドライバと同様に重力補償する。補償トルクはアクチュエータを通るので
    # actuator_force に含まれ、トルク上限の対象にもなる。
    base = spec.body("openarm_link0")
    base.pos = [0, 0, PEDESTAL_H]
    for b in spec.bodies:
        if b.name.startswith("openarm_"):
            b.gravcomp = 1.0
    for j in spec.joints:
        if j.name.startswith("openarm_"):
            j.actgravcomp = True

    hand = spec.body("openarm_hand")
    hand.add_geom(name="hand_box", type=mujoco.mjtGeom.mjGEOM_BOX,
                  size=[0.0285, 0.06, 0.013], pos=[0, 0, 0.0074],
                  rgba=[1, 0.3, 0.1, 0.3], group=3)
    hand.add_site(name="tcp", pos=[0, 0, 0.015 + PAD_POS_Z], size=[0.008] * 3,
                  rgba=[0, 1, 0, 1], group=4)
    for body, sign in (("openarm_right_finger", 1), ("openarm_left_finger", -1)):
        spec.body(body).add_geom(
            name=f"{body}_pad", type=mujoco.mjtGeom.mjGEOM_BOX, size=list(PAD_HALF),
            pos=[0, sign * (PAD_INNER_Y - PAD_HALF[1]), PAD_POS_Z],
            friction=[friction, 0.01, 0.001], condim=4, priority=1,
            solref=[0.01, 1], solimp=[0.95, 0.99, 0.001, 0.5, 2],
            rgba=[0.1, 0.8, 0.1, 1], group=3)

    world = spec.worldbody
    world.add_light(pos=[0, 0, 2], dir=[0, 0, -1], diffuse=[0.8] * 3)
    world.add_geom(name="table", type=mujoco.mjtGeom.mjGEOM_PLANE, size=[1, 1, 0.01],
                   rgba=[0.75, 0.7, 0.62, 1])
    world.add_geom(name="pedestal", type=mujoco.mjtGeom.mjGEOM_CYLINDER,
                   size=[0.06, PEDESTAL_H / 2, 0], pos=[0, 0, PEDESTAL_H / 2],
                   contype=0, conaffinity=0, rgba=[0.3, 0.3, 0.32, 1])
    world.add_camera(name="overview", pos=[1.1, 0, 0.75], xyaxes=[0, 1, 0, -0.5, 0, 0.87])
    for name, (x, y) in (("station_a", STATION_A), ("station_b", STATION_B)):
        world.add_site(name=name, pos=[x, y, 0.0005], size=[0.05, 0.05, 0.0005],
                       type=mujoco.mjtGeom.mjGEOM_BOX, rgba=[0.2, 0.4, 0.9, 0.4])

    box = world.add_body(name="box", pos=[*STATION_A, box_half])
    box.add_freejoint(name="box_free")
    box.add_geom(name="box_geom", type=mujoco.mjtGeom.mjGEOM_BOX, size=[box_half] * 3,
                 mass=box_mass, friction=[friction, 0.01, 0.001], condim=4,
                 rgba=[0.85, 0.55, 0.2, 1])

    # home キーフレームに箱の qpos を足す
    key = spec.keys[0]
    key.qpos = list(key.qpos) + [*STATION_A, box_half, 1, 0, 0, 0]
    return spec


def build_model(**kw):
    return build_spec(**kw).compile()

In [ ]:
%%writefile src/openarm_anomaly/scripted.py
"""差分 IK によるスクリプト制御のピック&運搬(CPU MuJoCo)。

RL 方策が収束しない場合の保険と、保持の成立確認に使う。
IK は「TCP 位置 + ハンドが真下を向く」の 5 自由度で解き、ヨーは自由にする
(手首の可動域が狭く、ヨーまで固定すると届く範囲がほぼ無くなるため)。
"""
import mujoco
import numpy as np
from scipy.optimize import least_squares

from .scene import STATION_A, STATION_B

N_ARM = 7
GRIP_OPEN, GRIP_CLOSE = 0.044, -0.005
DOWN = np.array([0.0, 0.0, -1.0])
HOVER = 0.15


def waypoints(box_half, hover=HOVER, speed=1.0):
    """(所要時間 s, tcp 目標位置, グリッパ指令) の列。"""
    ax, ay = STATION_A
    bx, by = STATION_B
    z_grasp = box_half
    z_hover = box_half + hover
    s = 1.0 / speed
    return [
        (1.0 * s, (ax, ay, z_hover), GRIP_OPEN),   # 接近
        (1.0 * s, (ax, ay, z_grasp), GRIP_OPEN),   # 降下
        (0.6, (ax, ay, z_grasp), GRIP_CLOSE),       # 把持
        (1.0 * s, (ax, ay, z_hover), GRIP_CLOSE),   # 持ち上げ
        (1.8 * s, (bx, by, z_hover), GRIP_CLOSE),   # 運搬
        (1.0 * s, (bx, by, z_grasp + 0.003), GRIP_CLOSE),  # 置く
        (0.5, (bx, by, z_grasp + 0.003), GRIP_OPEN),  # 解放
        (1.0 * s, (bx, by, z_hover), GRIP_OPEN),    # 退避
    ]


class Kinematics:
    """指令値側の関節角に対して運動学だけで IK を解く(ダイナミクスと切り離す)。"""

    def __init__(self, m):
        self.m = m
        self.d = mujoco.MjData(m)
        self.site = m.site("tcp").id
        self.jac_p = np.zeros((3, m.nv))
        self.jac_r = np.zeros((3, m.nv))
        self.lo = m.jnt_range[:N_ARM, 0]
        self.hi = m.jnt_range[:N_ARM, 1]

    def fk(self, q):
        self.d.qpos[:N_ARM] = q
        mujoco.mj_kinematics(self.m, self.d)
        mujoco.mj_comPos(self.m, self.d)
        return self.d.site_xpos[self.site].copy(), self.d.site_xmat[self.site].reshape(3, 3).copy()

    def solve(self, q, p_des, iters=3, max_step=0.1):
        """直前の解から追従する増分 IK(連続な軌道用)。"""
        q = q.copy()
        for _ in range(iters):
            p, R = self.fk(q)
            err = np.concatenate([p_des - p, np.cross(R[:, 2], DOWN)])
            mujoco.mj_jacSite(self.m, self.d, self.jac_p, self.jac_r, self.site)
            J = np.vstack([self.jac_p, self.jac_r])[:, :N_ARM]
            dq = J.T @ np.linalg.solve(J @ J.T + 1e-4 * np.eye(6), err)
            q = np.clip(q + np.clip(dq, -max_step, max_step), self.lo, self.hi)
        return q

    def solve_global(self, p_des, n_starts=20, seed=0):
        """マルチスタートの境界付き最小二乗(初期姿勢を求める用)。"""
        rng = np.random.default_rng(seed)

        def res(q):
            p, R = self.fk(q)
            return np.concatenate([p - p_des, 0.3 * (R[:, 2] - DOWN)])

        best = None
        for _ in range(n_starts):
            r = least_squares(res, rng.uniform(self.lo, self.hi), bounds=(self.lo, self.hi))
            if best is None or r.cost < best.cost:
                best = r
        return best.x, np.linalg.norm(best.fun)


def reset_to_ready(m, d, box_half):
    """ステーション A の上空で下向きの姿勢にし、箱のヨーをその時のハンドに合わせる。"""
    kin = Kinematics(m)
    q, err = kin.solve_global(np.array([*STATION_A, box_half + HOVER]))
    assert err < 5e-3, f"ready pose unreachable: err={err:.3f}"
    # 真下へ降ろした把持姿勢でのハンドのヨー
    q_g = kin.solve(q, np.array([*STATION_A, box_half]), iters=100)
    _, R = kin.fk(q_g)
    yaw = np.arctan2(R[1, 0], R[0, 0])
    mujoco.mj_resetDataKeyframe(m, d, 0)
    d.qpos[:N_ARM] = q
    d.ctrl[:N_ARM] = q
    jb = m.joint("box_free").qposadr[0]
    d.qpos[jb + 3:jb + 7] = [np.cos(yaw / 2), 0, 0, np.sin(yaw / 2)]
    mujoco.mj_forward(m, d)


class ScriptedController:
    def __init__(self, m, box_half, speed=1.0, ctrl_dt=0.02):
        self.m = m
        self.ctrl_dt = ctrl_dt
        self.kin = Kinematics(m)
        self.wps = waypoints(box_half, speed=speed)
        self.t_ends = np.cumsum([w[0] for w in self.wps])
        self.duration = self.t_ends[-1]
        self.q_cmd = None
        self.start = None

    def target(self, t):
        i = min(int(np.searchsorted(self.t_ends, t)), len(self.wps) - 1)
        t0 = self.t_ends[i - 1] if i > 0 else 0.0
        a = np.clip((t - t0) / self.wps[i][0], 0, 1)
        a = a * a * (3 - 2 * a)  # smoothstep
        p1 = np.array(self.wps[i][1])
        p0 = np.array(self.wps[i - 1][1]) if i > 0 else self.start
        return p0 + a * (p1 - p0), self.wps[i][2], i

    def __call__(self, d, t):
        if self.q_cmd is None:
            self.q_cmd = d.qpos[:N_ARM].copy()
            self.start = self.kin.fk(self.q_cmd)[0]
        p_des, grip, phase = self.target(t)
        self.q_cmd = self.kin.solve(self.q_cmd, p_des)
        d.ctrl[:N_ARM] = self.q_cmd
        d.ctrl[N_ARM] = grip
        return phase


def run_episode(m, d, ctrl, render=None, every=2, camera="overview"):
    """1 サイクル実行し、ログ(関節角・速度・指令・トルク・グリッパ・物体位置)を返す。"""
    n_sub = int(round(ctrl.ctrl_dt / m.opt.timestep))
    steps = int(ctrl.duration / ctrl.ctrl_dt) + 10
    log = {k: [] for k in ["qpos", "qvel", "ctrl", "torque", "grip", "box_pos", "tcp", "tcp_des", "phase"]}
    frames = []
    box = m.body("box").id
    site = m.site("tcp").id
    for k in range(steps):
        phase = ctrl(d, k * ctrl.ctrl_dt)
        for _ in range(n_sub):
            mujoco.mj_step(m, d)
        log["qpos"].append(d.qpos[:N_ARM].copy())
        log["qvel"].append(d.qvel[:N_ARM].copy())
        log["ctrl"].append(d.ctrl.copy())
        log["torque"].append(d.actuator_force[:N_ARM].copy())
        log["grip"].append(d.qpos[N_ARM])
        log["box_pos"].append(d.xpos[box].copy())
        log["tcp"].append(d.site_xpos[site].copy())
        log["tcp_des"].append(ctrl.target(k * ctrl.ctrl_dt)[0])
        log["phase"].append(phase)
        if render is not None and k % every == 0:
            render.update_scene(d, camera=camera)
            frames.append(render.render())
    return {k: np.array(v) for k, v in log.items()}, frames

In [ ]:
import json, os, sys, time
sys.path.insert(0, "src")
import jax, jax.numpy as jnp, mujoco, numpy as np
from mujoco import mjx
from openarm_anomaly.scene import STATION_B, build_model
from openarm_anomaly.scripted import ScriptedController, reset_to_ready, run_episode

print("jax", jax.__version__, jax.devices(), "mujoco", mujoco.__version__)
assert jax.devices()[0].platform == "gpu", "GPU が見えていない"
results = {"jax": jax.__version__, "device": str(jax.devices()[0]), "mujoco": mujoco.__version__}

## A. MJX(GPU)で保持が成立するか

CPU MuJoCo のスクリプト制御で指令列を記録し、同じ指令列を MJX で再生して箱の軌跡を比べる。
物理の時間刻み 0.002 s(基準)と 0.004 s(学習を速くできるか)の両方を、
`impl="jax"` と `impl="warp"`(MJWarp)で試す。Warp は接触数の上限を事前に確保するため、
上限が足りないと接触が落ちる。保持が CPU と一致するかで確かめる。

In [ ]:
HALF, MASS, CTRL_DT = 0.03, 1.5, 0.02
NACON_PER_ENV, NJMAX = 32, 256  # warp 用の接触数・拘束行数の上限


def batched_data(m, d0, impl, B):
    if impl == "jax":
        dx = mjx.put_data(m, d0)
        return jax.tree.map(lambda x: jnp.broadcast_to(x, (B,) + x.shape), dx)
    return jax.vmap(lambda i: mjx.put_data(
        m, d0, impl="warp", dummy_arg_for_batching=i,
        naconmax=B * NACON_PER_ENV, njmax=NJMAX))(jnp.arange(B))


def record(dt):
    m = build_model(box_half=HALF, box_mass=MASS, timestep=dt)
    d = mujoco.MjData(m)
    reset_to_ready(m, d, HALF)
    d0 = mujoco.MjData(m)
    d0.qpos[:], d0.ctrl[:] = d.qpos, d.ctrl
    mujoco.mj_forward(m, d0)
    log, _ = run_episode(m, d, ScriptedController(m, HALF, ctrl_dt=CTRL_DT))
    return m, d0, log


def make_rollout(m, mx, impl_tag=""):
    n_sub = int(round(CTRL_DT / m.opt.timestep))
    box_id = m.body("box").id

    def ctrl_step(dx, u):
        dx = dx.replace(ctrl=u)
        dx = jax.lax.fori_loop(0, n_sub, lambda _, x: mjx.step(mx, x), dx)
        return dx, dx.xpos[box_id]
    return ctrl_step, n_sub


results["replay"] = {}
for impl in ("jax", "warp"):
    for dt in (0.002, 0.004):
        key = f"{impl}_dt{dt}"
        try:
            m, d0, log = record(dt)
            mx = mjx.put_model(m, impl=impl)
            dxb = batched_data(m, d0, impl, 1)
            ctrl_step, n_sub = make_rollout(m, mx)
            roll = jax.jit(jax.vmap(lambda dx, us: jax.lax.scan(ctrl_step, dx, us)[1], in_axes=(0, None)))
            ctrls = jnp.asarray(log["ctrl"])
            t = time.time(); box = np.asarray(jax.block_until_ready(roll(dxb, ctrls)))[0]; t_compile = time.time() - t
            cpu_err = float(np.linalg.norm(log["box_pos"][-1, :2] - np.array(STATION_B)))
            mjx_err = float(np.linalg.norm(box[-1, :2] - np.array(STATION_B)))
            r = {"cpu_place_err": cpu_err, "cpu_lift": float(log["box_pos"][:, 2].max() - HALF),
                 "mjx_place_err": mjx_err, "mjx_lift": float(box[:, 2].max() - HALF),
                 "compile_s": t_compile}
            print(f"{key}: {json.dumps({k: round(v, 4) for k, v in r.items()})}")
        except Exception as e:
            r = {"error": f"{type(e).__name__}: {str(e)[:300]}"}
            print(f"{key}: FAILED {r['error']}")
        results["replay"][key] = r

## B. 並列速度(PPO の学習時間の見積もり)

同じ指令列を B 個の環境で並列に再生し、制御ステップ/秒を測る(方策の計算は含まない)。
PPO では 1 制御ステップ = 方策の 1 行動。`impl="jax"` と `impl="warp"`(MJWarp)を比べる。

In [ ]:
N_CTRL = 50  # 計測する制御ステップ数(1 秒分)
TARGET_STEPS = 100e6  # 見積もりの基準: PPO 1 億制御ステップ
results["throughput"] = []


def bench(dt, B, impl):
    m, d0, log = record(dt)
    mx = mjx.put_model(m, impl=impl)
    ctrl_step, n_sub = make_rollout(m, mx)
    ctrls = jnp.asarray(log["ctrl"][:N_CTRL])
    dxb = batched_data(m, d0, impl, B)
    f = jax.jit(jax.vmap(lambda x: jax.lax.scan(ctrl_step, x, ctrls)[1][-1]))
    t = time.time(); jax.block_until_ready(f(dxb)); t_compile = time.time() - t
    t = time.time(); last = np.asarray(jax.block_until_ready(f(dxb))); el = time.time() - t
    sps = B * N_CTRL / el
    r = {"impl": impl, "dt": dt, "B": B, "ctrl_steps_per_s": sps,
         "physics_steps_per_s": sps * n_sub, "compile_s": t_compile,
         "hours_per_100M": TARGET_STEPS / sps / 3600,
         "finite": bool(np.all(np.isfinite(last)))}
    results["throughput"].append(r)
    print(f"{impl:5s} dt={dt} B={B:5d}: {sps:10,.0f} ctrl steps/s  "
          f"-> {r['hours_per_100M']:.2f} h / 1e8 steps  (compile {t_compile:.0f}s, finite={r['finite']})")


for impl in ("jax", "warp"):
    for dt in (0.002, 0.004):
        for B in (256, 1024, 4096):
            try:
                bench(dt, B, impl)
            except Exception as e:  # warp が使えない、メモリ不足など
                print(f"{impl} dt={dt} B={B}: FAILED {type(e).__name__}: {str(e)[:300]}")
                results["throughput"].append({"impl": impl, "dt": dt, "B": B, "error": f"{type(e).__name__}: {str(e)[:300]}"})
                break

## C. ヘッドレス描画(EGL)と mp4 書き出し

In [ ]:
import mediapy
m = build_model(box_half=HALF, box_mass=1.0)
d = mujoco.MjData(m)
reset_to_ready(m, d, HALF)
cam = mujoco.MjvCamera()
cam.lookat[:], cam.distance, cam.azimuth, cam.elevation = [0.2, 0, 0.12], 1.0, 160, -22
r = mujoco.Renderer(m, 480, 640)
t = time.time()
log, frames = run_episode(m, d, ScriptedController(m, HALF), render=r, every=2, camera=cam)
t_render = time.time() - t
mediapy.write_video("outputs/pick_place_kaggle.mp4", frames, fps=25)
results["render"] = {"frames": len(frames), "seconds": t_render, "mp4_bytes": os.path.getsize("outputs/pick_place_kaggle.mp4")}
print(results["render"])
mediapy.show_video(frames, fps=25)

In [ ]:
with open("/kaggle/working/day1_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))